# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [ ]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, 
                      n = 30, max_steps = 100, visualize = False, animation = False):    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [27]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [28]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [29]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [30]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('east')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [31]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[3 3]'; Hunter 2 is at '[2 3]'; Monster is at '[3 0]'

.....
.....
...2.
M..1.
.....
Hunter 1 chose action 'east'
Hunter 2 chose action 'south'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[3 4]'; Hunter 2 is at '[3 3]'; Monster is at '[3 0]'

.....
.....
.....
M..21
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'east'
Monster chose action 'west'

=====

Step 2:
Hunter 1 is at '[3 3]'; Hunter 2 is at '[3 4]'; Monster is at '[3 0]'

.....
.....
.....
M..12
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'south'
Monster chose action 'south'

=====

Step 3:
Hunter 1 is at '[3 2]'; Hunter 2 is at '[4 4]'; Monster is at '[4 0]'

.....
.....
.....
..1..
M...2
Hunter 1 chose action 'north'
Hunter 2 chose action 'north'
Monster chose action 'stay'

=====

Step 4:
Hunter 1 is at '[2 2]'; Hunter 2 is at '[3 4]'; Monster is at '[4 0]'

.....
.....
..1..
....2
M....
Hunter 1 chose action 'west'
Hunter 2 chose action 'east'
Monster chose ac

('M', nan)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [32]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('2', 17), ('M', nan), ('2', 52), ('M', nan), ('2', 49), ('M', nan), ('M', nan), ('M', nan), ('1', 10), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 71), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 70), ('M', nan), ('M', nan), ('M', nan), ('1', 42), ('M', nan), ('1', 39), ('1', 59), ('M', nan), ('1', 46), ('M', nan), ('M', nan), ('M', nan), ('2', 1), ('1', 16), ('M', nan), ('M', nan), ('M', nan), ('1', 2), ('M', nan), ('2', 63), ('M', nan), ('M', nan), ('1', 35), ('M', nan), ('M', nan), ('M', nan), ('1', 1), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 44), ('M', nan), ('1', 7), ('M', nan), ('M', nan), ('1', 22), ('M', nan), ('2', 45), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 58), ('1', 76), ('M', nan), ('M', nan), ('M', nan), ('2', 70), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 36), ('M', nan), ('M', n

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [33]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 18
Hunter 2 won: 11
Monster 1 won: 71


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [ ]:
# here goes your agent implementation
def rule_based_hunter_agent_function(hunter_location, monster_location):
    
    #Calculate the distance
    x = monster_location[1] - hunter_location[1]
    y = monster_location[0] - hunter_location[0]

    #Prioritize moving along the axis with the larger distance
    if abs(x) >= abs(y):
        if x < 0:
            return 'west'
        elif x > 0:
            return 'east'
    else:
        if y < 0:
            return 'north'
        elif y > 0:
            return 'south'
    #Use teleport ONLY for fallback action
    return 'teleport'


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [35]:
step = arena_environment(simple_randomized_hunter_agent_function,rule_based_hunter_agent_function,monster_agent_function_simple,
                         n=30,max_steps=500,visualize=True,animation=True)

Step 18:
Hunter 1 is at '[12  6]'; Hunter 2 is at '[15  6]'; Monster is at '[15  6]'

..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
......1.......................
..............................
..............................
......2.......................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
...............

In [36]:
# Your experimentation code goes here
# Your experimentation code goes here
grid_size = 30
max_simulation_steps = 500
runs = 100
results = []
for _ in range(runs):
    winner,steps = arena_environment(simple_randomized_hunter_agent_function,rule_based_hunter_agent_function,monster_agent_function_simple,n=grid_size,max_steps=max_simulation_steps,visualize=False,animation=False) #Tắt Visualize và animation để ra thẳng kết quả.
    results.append((winner,steps))
results_arr = np.array(results)
winners = results_arr[:,0]
steps_taken = results_arr[:,1].astype(float)
print(f"Hunter 1 (Random) won: {np.sum(winners == '1')} times.")
print(f"Hunter 2 (Rule-based) won: {np.sum(winners == '2')} times.")
print(f"Monster won: {np.sum(winners == 'M')} times.")
successful_steps = steps_taken[~np.isnan(steps_taken)]
if len(successful_steps) > 0:
    print(f"Average steps to catch the monster: {round(np.mean(successful_steps), 2)} steps!")

Hunter 1 (Random) won: 2 times.
Hunter 2 (Rule-based) won: 98 times.
Monster won: 0 times.
Average steps to catch the monster: 24.73 steps!


## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

1. Thợ săn tính toán khoảng cách giữa vị trí hiện tại của nó và vị trí của quái vật. Sau đó, nó chọn di chuyển dọc theo trục có khoảng cách lớn hơn. Chiến thuật này hoạt động đặc biệt hiệu quả vì nó đảm bảo thợ săn luôn đi theo quỹ đạo trực tiếp và tối ưu nhất hướng tới mục tiêu mà không lãng phí bất kỳ bước đi nào.

2. Không, hành động dịch chuyển tức thời không được sử dụng, chỉ đóng vai trò là lệnh dự phòng. Dịch chuyển đưa agents đến một tọa độ hoàn toàn ngẫu nhiên, làm triệt tiêu hoàn toàn lợi thế của thuật toán tìm khoảng cách. Việc bước đi từng bước đảm bảo khoảng cách luôn được thu hẹp, trong khi dịch chuyển hoàn toàn phụ thuộc vào yếu tố may mắn.

3. Hiệu suất tỷ lệ thuận với kích thước đấu trường. Đấu trường lớn hơn làm tăng khoảng cách ban đầu giữa thợ săn và quái vật, dẫn đến số bước trung bình cần để bắt quái vật cao hơn. Tuy nhiên, tỷ lệ chiến thắng của agent vẫn duy trì ở mức gần 100% vì logic toán học luôn thu hẹp khoảng cách bất kể bản đồ có kích thước bao nhiêu.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

Thay đổi thuật toán cho Quái vật:
Trong code hiện tại, hàm monster_agent_function_simple nhận trực tiếp tọa độ của cả hunter_pos và monster_pos trong mọi lượt đi mà không cần phải tốn bước để "listen". Thế nhưng quái vật lại chỉ đi random và có xác suất $50\%$ đứng im.

Để quái vật sống sót lâu hơn, ta có thể cải thiện nó thành một Reflex Agent thông minh:

- Chạy trốn thay vì đứng im: Loại bỏ hoàn toàn hành động "stay" (xác suất $50\%$). Việc đứng im chỉ giúp thợ săn bắt kịp nhanh hơn.

- Đi theo hướng ngược lại: Quái vật nên tính toán sự chênh lệch tọa độ tương tự như thợ săn. Thay vì thu hẹp khoảng cách, nó phải chọn hành động sao cho khoảng cách trục x,y ngày càng lớn ra.

Né góc: Khi quái vật chạm biên của map (VD monster_pos[0] == 0 là vướng tường), nó cần biết đánh lái qua lại thay vì cố gắng đi về hướng tường và bị kẹt, tạo cơ hội cho thợ săn áp sát dồn vào chân tường.

# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.